
```python
'''Recording trajectories and storing them into a databaseself.

conda activate gesturenlu2
cd ~/lfd_ws/
source install/setup.bash
ros2 launch object_localization box_localization_launch.py

conda activate gesturenlu2
cd ~/lfd_ws/
source install/setup.bash
ros2 run gesture_detector leap

sudo leapd
'''
```

```
conda activate gesturenlu2
cd ~/lfd_ws/
source install/setup.bash
python /home/imitlearn/lfd_ws/src/nocode_robot_programming/main.py

conda activate gesturenlu2
cd ~/lfd_ws/
source install/setup.bash
cd src
code .
```

WARNING!!! When restarting the Jupyter kernel, you might need to kill it's residual proceess with: `pkill -9 -f ipykernel_launcher`

---

GESTURE TELEOP LIKES FORCES = 100 N/m
KINESTHETIC TEACHING = 0 N/m
EXECUTION = 1000 N/m
JOYSTICK = 1000 N/m

# Timeline

- [ ] Test replay 
- [x] See the task-graph structure
- [ ] Get the context dataset for OSE
- [ ] 


# Initialization

In [ ]:
from nocode_robot_programming.state_decision.state_decider import StateDeciderBase
from nocode_robot_programming.state_decision.dataloader import TrajectoryDataset
import trajectory_data

In [ ]:
dataset = TrajectoryDataset(trajectory_data.package_path)
d = dataset.get_image_dataset(dataset.get_all_names("user_0_kine_peg_pick"))
decider = StateDeciderBase()
decider.train(d.X, d.y_int)

In [ ]:
import rclpy
rclpy.init()
from skills_manager.risk_aware_lfd.ralfd import RALfD
from skills_manager.ros_param_manager import set_remote_parameters
from nocode_robot_programming.state_decision.dino_model import DINOStateDecider


lfd = RALfD(state_decider=decider)
lfd.start()
lfd.keyboard_start()
lfd.joy_start()
lfd.teleop_start()
set_remote_parameters(lfd, ["position_x", "position_y", "position_z", "orientation_x", "orientation_y", "orientation_z", "orientation_w"],
[0.4, 0.0, 0.4, 1.0, 0.0, 0.0, 0.0], server="localizer_node")

In [ ]:
lfd.K_pos = 100
lfd.K_ori = 120
lfd.set_stiffness(lfd.K_pos,lfd.K_pos,lfd.K_pos,lfd.K_ori,lfd.K_ori,lfd.K_ori,0)
lfd.traj_rec()

# 1. Name a new task

In [ ]:
name_skill = "user_0_kine_peg_pick"
name_skill = lfd.declare_parameter_and_get('name_skill', name_skill)
f"Name skill: {name_skill}, {'Exists' if lfd.skill_exists(name_skill) else 'Not Exists'}"

# 2. Move to start

In [ ]:
lfd.K_pos = 1000
lfd.move_template_start()
lfd.K_pos = 100

# 3. Check if unique & Demonstrate

In [ ]:
if lfd.skill_exists(name_skill):
    raise Exception("Skill Exists!")
lfd.traj_rec()
lfd.save(name_skill)

# 4. Move to start & Replay skill

In [ ]:
lfd.K_pos = 1000
lfd.move_template_start()
request = lfd.play_skill(name_skill, None, localize_box=False)

if isinstance(request, tuple):
    lfd.K_pos = 100
    lfd.traj_rec()
    lfd.save(request[2])

In [ ]:
lfd.state_decider.train()

In [ ]:
lfd.state_decider.predict(lfd.get_observations()[0], lfd.time_index)

In [ ]:
rclpy.shutdown()

In [ ]:
lfd.desk.lock()

In [ ]:
lfd.desk.unlock()

In [ ]:
lfd.home_gripper() # fix gripper moving

In [ ]:
lfd.grasp_gripper(0.0) # sometimes need to be called multiple times

In [ ]:
lfd.input_check() # check in case input not received